# Interview Integrity Monitor — 1-second photo pipeline

This notebook implements a **photo-at-a-time** monitoring pipeline for a candidate screening session.

**Pipeline:** camera/video capture → image analysis → 5-second batch review signal → saved evidence and dashboard data.

### What it does
- Calibrates a person's normal head/eye landmark ratios from about **11–12 photos during the first ~8 seconds** (calibration capture interval is faster than the normal 1-second interval).
- After calibration, captures one photo per second.
- Checks face/person presence, approximate gaze/head direction, and supported electronic devices (phone/laptop; tablet requires a custom YOLO model with a `tablet` class).
- Uses **three worker threads**: capture, per-photo analysis, and 5-second batch scoring.
- Saves timestamped observations, annotated evidence photos, and batch summaries for a Streamlit dashboard.
- Shows “Please look at the camera” when cues persist; this is a prompt, **not a verdict**.

> Important: head/gaze direction and object detections are imperfect signals. A flag means **review needed**, not proof of cheating. Do not make high-stakes decisions without human review and validation on representative recordings.

### Run order
1. Install dependencies in the next cell (or install them in your environment).
2. Run setup / class cells.
3. Run the pipeline cell. It starts the three threads and writes `dashboard_app.py`.
4. In a terminal from the notebook directory, run `streamlit run dashboard_app.py` to open the dashboard.

In [1]:
# If you are in a fresh environment, uncomment and run this cell once.
# Use a MediaPipe version that exposes mp.solutions.face_mesh.
# %pip install "opencv-python>=4.8" "mediapipe>=0.10,<0.11" "ultralytics>=8.2" streamlit pandas pillow pyyaml

import cv2
import json
import time
import math
import threading
import queue
import logging
from pathlib import Path
from datetime import datetime, timezone
from collections import deque
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import mediapipe as mp
from ultralytics import YOLO

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("interview_monitor")


## 1. Settings

The defaults aim to keep this prototype simple. Update `VIDEO_SOURCE` to `0` for webcam or to a video path for repeatable testing. `OUTPUT_DIR` stores evidence and dashboard data.

In [2]:
# -------------------- USER SETTINGS --------------------
VIDEO_SOURCE = 0                 # 0 = webcam; or e.g. "data/sample/test.mp4"
OUTPUT_DIR = Path("interview_monitor_output")
YOLO_WEIGHTS = "yolo11n.pt"      # COCO model detects person, cell phone, laptop; not tablets reliably

NORMAL_CAPTURE_SECONDS = 1.0     # after calibration: capture one photo each second
CALIBRATION_SECONDS = 8.0        # calibration window
CALIBRATION_SAMPLES_TARGET = 12  # target baseline photos; capture interval is shortened during calibration
CALIBRATION_CAPTURE_SECONDS = 0.75
BATCH_SECONDS = 5.0               # batch evaluation cadence
FACE_MISSING_FLAG = True
SUSPICION_THRESHOLD = 2           # number of distinct cue types in a 5-second batch
MIN_OBJECT_CONFIDENCE = 0.35
MIN_PERSON_CONFIDENCE = 0.40

# Head/gaze deviation thresholds in normalized landmark-ratio units.
# Tune these only after reviewing validation recordings.
THRESHOLDS = {"iris_x": 0.12, "iris_y": 0.12, "head_x": 0.10, "head_y": 0.10}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for sub in ("photos", "annotated", "batches"):
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)


## 2. Data structures and baseline calibration

`PhotoJob` is a captured photo waiting for analysis. `PhotoResult` is the structured output for one photo. The calibration baseline is the median of the first valid landmark measurements, which reduces the impact of one noisy frame.

In [3]:
@dataclass
class PhotoJob:
    captured_at: str
    monotonic_s: float
    frame: np.ndarray
    index: int

@dataclass
class PhotoResult:
    captured_at: str
    index: int
    face_present: bool
    face_count: int
    person_count: int
    gaze_direction: str
    head_direction: str
    head_deviation: float | None
    gaze_deviation: float | None
    devices: list
    cues: list
    calibration_complete: bool
    annotated_path: str
    photo_path: str

class HeadGazeCalibrator:
    """Collects normal eye/head ratios and compares future photos to their median."""
    KEYS = ("iris_x", "iris_y", "head_x", "head_y")

    def __init__(self, target_samples=CALIBRATION_SAMPLES_TARGET, max_seconds=CALIBRATION_SECONDS):
        self.target_samples = int(target_samples)
        self.max_seconds = float(max_seconds)
        self.started_at = None
        self.samples = []
        self.baseline = None
        self.complete = False

    def add(self, metrics, elapsed_s):
        """Add one valid face sample during calibration; return True once baseline is ready."""
        if self.complete:
            return True
        if self.started_at is None:
            self.started_at = elapsed_s
        if metrics is not None:
            self.samples.append([float(metrics[k]) for k in self.KEYS])
        elapsed = elapsed_s - self.started_at
        # Finish after the calibration window and at least 5 valid samples; or once target is reached.
        if (elapsed >= self.max_seconds and len(self.samples) >= 5) or len(self.samples) >= self.target_samples:
            med = np.median(np.asarray(self.samples, dtype=float), axis=0)
            self.baseline = dict(zip(self.KEYS, med.astype(float).tolist()))
            self.complete = True
            log.info("Calibration complete with %d valid face samples", len(self.samples))
        return self.complete

    def compare(self, metrics):
        """Return approximate gaze direction, head direction, and normalized deviations."""
        if not self.complete or metrics is None or self.baseline is None:
            return "CALIBRATING", "CALIBRATING", None, None
        dx = (metrics["head_x"] - self.baseline["head_x"]) / max(THRESHOLDS["head_x"], 1e-6)
        dy = (metrics["head_y"] - self.baseline["head_y"]) / max(THRESHOLDS["head_y"], 1e-6)
        gx = (metrics["iris_x"] - self.baseline["iris_x"]) / max(THRESHOLDS["iris_x"], 1e-6)
        gy = (metrics["iris_y"] - self.baseline["iris_y"]) / max(THRESHOLDS["iris_y"], 1e-6)
        head_mag = max(abs(dx), abs(dy))
        gaze_mag = max(abs(gx), abs(gy))
        head_dir = "CENTER" if head_mag < 1 else ("RIGHT" if dx > 0 else "LEFT") if abs(dx) >= abs(dy) else ("DOWN" if dy > 0 else "UP")
        gaze_dir = "CENTER" if gaze_mag < 1 else ("RIGHT" if gx > 0 else "LEFT") if abs(gx) >= abs(gy) else ("DOWN" if gy > 0 else "UP")
        return gaze_dir, head_dir, float(head_mag), float(gaze_mag)


## 3. Per-photo computer vision

MediaPipe Face Mesh provides approximate iris/head ratios. YOLO detects people and supported devices. The detector model is loaded once and reused by the analysis thread. Since inference is performed by only one worker thread, the models are not called concurrently.

In [4]:
def project_ratio(point, start, end):
    """Project a landmark onto an eye/face axis; roughly 0.5 means centered."""
    axis = end - start
    return float(np.dot(point - start, axis) / (np.dot(axis, axis) + 1e-9))

def get_face_metrics(points):
    """Return coarse eye and head ratios from normalized Face Mesh landmarks."""
    return {
        "iris_x": (project_ratio(points[468], points[33], points[133]) + project_ratio(points[473], points[362], points[263])) / 2,
        "iris_y": (project_ratio(points[468], points[159], points[145]) + project_ratio(points[473], points[386], points[374])) / 2,
        "head_x": project_ratio(points[1], points[234], points[454]),
        "head_y": project_ratio(points[1], points[10], points[152]),
    }

class PhotoAnalyzer:
    """Analyzes one captured image and returns detections plus an annotated copy."""
    def __init__(self, yolo_weights=YOLO_WEIGHTS):
        self.face_mesh = None
        self.yolo = None
        try:
            self.face_mesh = mp.solutions.face_mesh.FaceMesh(
                static_image_mode=True, max_num_faces=5, refine_landmarks=True,
                min_detection_confidence=0.5)
        except Exception as exc:
            log.exception("Could not initialize MediaPipe Face Mesh: %s", exc)
        try:
            self.yolo = YOLO(yolo_weights)
        except Exception as exc:
            log.exception("Could not load YOLO weights %s: %s", yolo_weights, exc)

    def analyze(self, job, calibrator, elapsed_s):
        frame = job.frame.copy()
        height, width = frame.shape[:2]
        cues, devices = [], []
        face_present, face_count = False, 0
        metrics = None
        gaze_direction, head_direction = "UNKNOWN", "UNKNOWN"
        head_dev = gaze_dev = None
        # Face and gaze/head analysis
        if self.face_mesh is not None:
            try:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                result = self.face_mesh.process(rgb)
                faces = result.multi_face_landmarks or []
                face_count = len(faces)
                face_present = face_count > 0
                if faces:
                    # Use the largest face as the candidate face.
                    pts_all = []
                    for face in faces:
                        pts = np.array([(p.x, p.y) for p in face.landmark], dtype=np.float32)
                        area = max(0.0, float(np.ptp(pts[:, 0]) * np.ptp(pts[:, 1])))
                        pts_all.append((area, pts))
                    _, points = max(pts_all, key=lambda item: item[0])
                    metrics = get_face_metrics(points) if len(points) > 473 else None
                    x1 = int(np.clip(points[:, 0].min() * width, 0, width - 1))
                    y1 = int(np.clip(points[:, 1].min() * height, 0, height - 1))
                    x2 = int(np.clip(points[:, 0].max() * width, 0, width - 1))
                    y2 = int(np.clip(points[:, 1].max() * height, 0, height - 1))
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (60, 200, 60), 2)
                    cv2.putText(frame, "Face", (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (60, 200, 60), 2)
            except Exception as exc:
                log.warning("Face analysis failed for photo %s: %s", job.index, exc)
        if not face_present:
            cues.append("face_missing")
        if face_count > 1:
            cues.append("multiple_faces")

        calibration_ready = calibrator.add(metrics, elapsed_s) if not calibrator.complete else True
        if calibrator.complete and metrics is not None:
            gaze_direction, head_direction, head_dev, gaze_dev = calibrator.compare(metrics)
            if head_dev is not None and head_dev >= 1.0:
                cues.append("head_turned")
            if gaze_dev is not None and gaze_dev >= 1.0:
                cues.append("gaze_away")

        # Person and electronic-device detection. COCO class names include person, cell phone, laptop.
        person_count = 0
        if self.yolo is not None:
            try:
                yres = self.yolo.predict(frame, imgsz=640, conf=MIN_OBJECT_CONFIDENCE, verbose=False)[0]
                if yres.boxes is not None:
                    for box in yres.boxes:
                        class_id = int(box.cls.item())
                        name = str(self.yolo.names.get(class_id, class_id)).lower()
                        conf = float(box.conf.item())
                        if name == "person" and conf < MIN_PERSON_CONFIDENCE:
                            continue
                        if name != "person" and name in ("cell phone", "laptop", "tablet") and conf < MIN_OBJECT_CONFIDENCE:
                            continue
                        x1, y1, x2, y2 = [int(v) for v in box.xyxy[0].tolist()]
                        if name == "person":
                            person_count += 1
                            color = (255, 160, 0)
                        elif name in ("cell phone", "laptop", "tablet"):
                            devices.append({"name": name, "confidence": round(conf, 3), "bbox": [x1, y1, x2, y2]})
                            color = (0, 0, 255)
                        else:
                            continue
                        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                        cv2.putText(frame, f"{name} {conf:.2f}", (x1, max(20, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
            except Exception as exc:
                log.warning("Object detection failed for photo %s: %s", job.index, exc)
        if person_count > 1:
            cues.append("second_person")
        for device in devices:
            cues.append("device_" + device["name"].replace(" ", "_"))

        # A clear on-screen prompt is overlaid whenever the current image has a concern cue.
        prompt = "CALIBRATING: look at the camera" if not calibrator.complete else (
            "Please look at the camera" if any(c in cues for c in ("head_turned", "gaze_away", "face_missing", "multiple_faces")) else "Monitoring")
        cv2.rectangle(frame, (0, 0), (width, 38), (25, 25, 25), -1)
        cv2.putText(frame, prompt, (12, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.68,
                    (0, 210, 255) if "Please" in prompt or "CALIBRATING" in prompt else (80, 220, 80), 2)

        stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        photo_path = OUTPUT_DIR / "photos" / f"photo_{job.index:06d}_{stamp}.jpg"
        annotated_path = OUTPUT_DIR / "annotated" / f"annotated_{job.index:06d}_{stamp}.jpg"
        cv2.imwrite(str(photo_path), job.frame)
        cv2.imwrite(str(annotated_path), frame)
        return PhotoResult(
            captured_at=job.captured_at, index=job.index, face_present=face_present,
            face_count=face_count, person_count=person_count, gaze_direction=gaze_direction,
            head_direction=head_direction, head_deviation=head_dev, gaze_deviation=gaze_dev,
            devices=devices, cues=sorted(set(cues)), calibration_complete=calibrator.complete,
            annotated_path=str(annotated_path), photo_path=str(photo_path))

    def close(self):
        if self.face_mesh is not None:
            try: self.face_mesh.close()
            except Exception: pass


## 4. Thread-safe three-worker pipeline

- **Thread 1 — capture:** captures one frame on a timer and puts it in a bounded queue.
- **Thread 2 — photo analysis:** consumes each captured photo and runs face/gaze/person/device detection.
- **Thread 3 — batch analysis:** every five seconds reviews the analyzed photos from that interval and writes a batch result.

Queues are bounded so a slow detector does not cause unlimited memory growth. If the analysis queue is full, the oldest waiting photo is dropped and recorded in logs. This is a prototype: monitor dropped-photo counts and processing latency before production use.

In [5]:
class InterviewPipeline:
    def __init__(self, source=VIDEO_SOURCE):
        self.source = source
        self.photo_queue = queue.Queue(maxsize=20)
        self.result_queue = queue.Queue(maxsize=100)
        self.stop_event = threading.Event()
        self.lock = threading.Lock()
        self.results = deque(maxlen=5000)
        self.batches = deque(maxlen=1000)
        self.status = {"running": False, "calibration_complete": False, "last_message": "Not started", "dropped_photos": 0}
        self.started_mono = None
        self.frame_index = 0
        self.analyzer = None
        self.calibrator = HeadGazeCalibrator()
        self.threads = []
        self.capture = None
        self._batch_results = []
        self._last_batch_mono = None
        self._dashboard_path = Path("dashboard_app.py")
        self._write_dashboard_app()

    def _append_jsonl(self, filename, record):
        with (OUTPUT_DIR / filename).open("a", encoding="utf-8") as f:
            f.write(json.dumps(record, default=str) + "\n")

    def _capture_worker(self):
        """Worker 1: capture images at the configured cadence."""
        self.capture = cv2.VideoCapture(self.source)
        if not self.capture.isOpened():
            with self.lock:
                self.status["last_message"] = f"Could not open video source: {self.source}"
            self.stop_event.set()
            return
        while not self.stop_event.is_set():
            now = time.monotonic()
            elapsed = now - self.started_mono
            interval = CALIBRATION_CAPTURE_SECONDS if not self.calibrator.complete and elapsed < CALIBRATION_SECONDS else NORMAL_CAPTURE_SECONDS
            ok, frame = self.capture.read()
            if not ok:
                with self.lock:
                    self.status["last_message"] = "Video ended or frame could not be read"
                self.stop_event.set()
                break
            self.frame_index += 1
            job = PhotoJob(datetime.now(timezone.utc).isoformat(), now, frame.copy(), self.frame_index)
            try:
                self.photo_queue.put(job, timeout=0.25)
            except queue.Full:
                # Prefer recent evidence: remove one stale waiting photo, then enqueue current photo.
                try:
                    self.photo_queue.get_nowait()
                    self.photo_queue.task_done()
                    with self.lock:
                        self.status["dropped_photos"] += 1
                except queue.Empty:
                    pass
                try: self.photo_queue.put_nowait(job)
                except queue.Full: pass
            time.sleep(interval)
        if self.capture is not None:
            self.capture.release()

    def _analysis_worker(self):
        """Worker 2: one-photo-at-a-time detection and feature extraction."""
        self.analyzer = PhotoAnalyzer(YOLO_WEIGHTS)
        while not self.stop_event.is_set() or not self.photo_queue.empty():
            try:
                job = self.photo_queue.get(timeout=0.25)
            except queue.Empty:
                continue
            try:
                elapsed = job.monotonic_s - self.started_mono
                result = self.analyzer.analyze(job, self.calibrator, elapsed)
                row = asdict(result)
                self._append_jsonl("observations.jsonl", row)
                with self.lock:
                    self.results.append(row)
                    self.status["calibration_complete"] = self.calibrator.complete
                    self.status["last_message"] = "Photo analyzed"
                try: self.result_queue.put_nowait(row)
                except queue.Full:
                    with self.lock:
                        self.status["last_message"] = "Result queue full; some batch updates may be delayed"
            except Exception as exc:
                log.exception("Photo analysis error: %s", exc)
            finally:
                self.photo_queue.task_done()
        if self.analyzer is not None:
            self.analyzer.close()

    def _batch_worker(self):
        """Worker 3: every BATCH_SECONDS aggregate recent photo results and write review flags."""
        self._last_batch_mono = time.monotonic()
        while (not self.stop_event.is_set() or not self.result_queue.empty() or
               (len(self.threads) > 1 and self.threads[1].is_alive())):
            try:
                row = self.result_queue.get(timeout=0.25)
                self._batch_results.append(row)
                self.result_queue.task_done()
            except queue.Empty:
                pass
            now = time.monotonic()
            if now - self._last_batch_mono >= BATCH_SECONDS and self._batch_results:
                rows, self._batch_results = self._batch_results, []
                cue_counts = {}
                for item in rows:
                    for cue in item.get("cues", []):
                        cue_counts[cue] = cue_counts.get(cue, 0) + 1
                # Count cue types observed in the batch, rather than claiming a calibrated probability.
                meaningful = {k: v for k, v in cue_counts.items() if v >= 1}
                distinct_types = len(meaningful)
                flag = distinct_types >= SUSPICION_THRESHOLD
                batch = {
                    "created_at": datetime.now(timezone.utc).isoformat(),
                    "window_seconds": BATCH_SECONDS,
                    "photos_analyzed": len(rows),
                    "cue_counts": meaningful,
                    "review_flag": bool(flag),
                    "status": "REVIEW NEEDED" if flag else "NO MULTI-CUE FLAG",
                    "note": "Heuristic batch indicator; not a probability or proof of misconduct."
                }
                self._append_jsonl("batches.jsonl", batch)
                with self.lock:
                    self.batches.append(batch)
                    self.status["last_message"] = batch["status"]
                self._batch_results = []
                self._last_batch_mono = now
        # Flush a final partial batch when the source ends.
        if self._batch_results:
            rows, self._batch_results = self._batch_results, []
            cue_counts = {}
            for item in rows:
                for cue in item.get("cues", []): cue_counts[cue] = cue_counts.get(cue, 0) + 1
            meaningful = {k: v for k, v in cue_counts.items() if v >= 1}
            batch = {"created_at": datetime.now(timezone.utc).isoformat(), "window_seconds": BATCH_SECONDS,
                     "photos_analyzed": len(rows), "cue_counts": meaningful,
                     "review_flag": len(meaningful) >= SUSPICION_THRESHOLD,
                     "status": "REVIEW NEEDED" if len(meaningful) >= SUSPICION_THRESHOLD else "NO MULTI-CUE FLAG",
                     "note": "Final partial batch; heuristic only."}
            self._append_jsonl("batches.jsonl", batch)
            with self.lock: self.batches.append(batch)

    def start(self):
        """Start the three workers. Call stop() to end the session."""
        if self.status["running"]:
            return
        self.started_mono = time.monotonic()
        self.stop_event.clear()
        with self.lock:
            self.status["running"] = True
            self.status["last_message"] = "Workers started"
        self.threads = [
            threading.Thread(target=self._capture_worker, name="photo-capture", daemon=True),
            threading.Thread(target=self._analysis_worker, name="photo-analysis", daemon=True),
            threading.Thread(target=self._batch_worker, name="five-second-batch", daemon=True),
        ]
        for thread in self.threads: thread.start()
        log.info("Started capture, analysis, and batch threads")

    def stop(self):
        """Stop all workers and wait briefly for queued photos to finish."""
        self.stop_event.set()
        for thread in self.threads:
            thread.join(timeout=10)
        with self.lock:
            self.status["running"] = False
            self.status["last_message"] = "Stopped"
        log.info("Pipeline stopped")

    def snapshot(self):
        with self.lock:
            return {"status": dict(self.status), "results": list(self.results), "batches": list(self.batches)}

    def _write_dashboard_app(self):
        """Create a small Streamlit dashboard file from this notebook so the deliverable stays one notebook."""
        dashboard_source = r"""
import json, time
from pathlib import Path
import pandas as pd
import streamlit as st

OUT = Path("interview_monitor_output")
st.set_page_config(page_title="Interview Monitoring Dashboard", layout="wide")
st.title("Interview Monitoring Dashboard")
st.caption("Photo-based review signals. Flags are heuristic and require human review.")

def read_jsonl(path):
    if not path.exists(): return []
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            try: rows.append(json.loads(line))
            except json.JSONDecodeError: continue
    return rows

observations = read_jsonl(OUT / "observations.jsonl")
batches = read_jsonl(OUT / "batches.jsonl")
latest = observations[-1] if observations else None
c1, c2, c3, c4 = st.columns(4)
c1.metric("Photos analyzed", len(observations))
c2.metric("Batches", len(batches))
c3.metric("Review flags", sum(bool(b.get("review_flag")) for b in batches))
c4.metric("Latest cue count", len(latest.get("cues", [])) if latest else 0)

if latest:
    st.subheader("Latest analyzed photo")
    left, right = st.columns([1.2, 1])
    with left:
        img = latest.get("annotated_path")
        if img and Path(img).exists(): st.image(img, caption=latest.get("captured_at", ""), use_container_width=True)
    with right:
        st.write("**Face present:**", latest.get("face_present"))
        st.write("**Face count:**", latest.get("face_count"))
        st.write("**Person count:**", latest.get("person_count"))
        st.write("**Gaze direction (approx.):**", latest.get("gaze_direction"))
        st.write("**Head direction (approx.):**", latest.get("head_direction"))
        st.write("**Detected devices:**", latest.get("devices", []))
        st.write("**Current cues:**", latest.get("cues", []))

st.subheader("5-second batch review")
if batches:
    bdf = pd.DataFrame(batches)
    st.dataframe(bdf.iloc[::-1], use_container_width=True, hide_index=True)
else: st.info("No batch results yet. Start the notebook pipeline and wait for a batch window.")

st.subheader("Photo observations")
if observations:
    odf = pd.DataFrame(observations)
    cols = [c for c in ["captured_at", "index", "face_present", "face_count", "person_count", "gaze_direction", "head_direction", "devices", "cues"] if c in odf]
    st.dataframe(odf[cols].iloc[::-1], use_container_width=True, hide_index=True)

if st.button("Refresh dashboard"):
    st.rerun()
time.sleep(2)
st.rerun()
"""
        self._dashboard_path.write_text(dashboard_source, encoding="utf-8")
        log.info("Streamlit dashboard source written to %s", self._dashboard_path.resolve())


## 5. Start the pipeline

Run this cell once. For a webcam, grant camera access. For reproducible testing, use a video file path instead of `0`. The notebook writes the dashboard script automatically; launch it in a terminal using the command shown below.

In [ ]:
# Create and start the pipeline. The three workers run in background threads.
pipeline = InterviewPipeline(source=VIDEO_SOURCE)
pipeline.start()

print("Pipeline started.")
print("Dashboard file:", Path("dashboard_app.py").resolve())
print("When ready, run in a terminal: streamlit run dashboard_app.py")
print("Evidence output:", OUTPUT_DIR.resolve())
print("To stop the session, run: pipeline.stop()")


2026-10-12 14:13:33,330 | INFO | Streamlit dashboard source written to D:\claude\interview\dashboard_app.py
2026-10-12 14:13:33,351 | INFO | Started capture, analysis, and batch threads


Pipeline started.
Dashboard file: D:\claude\interview\dashboard_app.py
When ready, run in a terminal: streamlit run dashboard_app.py
Evidence output: D:\claude\interview\interview_monitor_output
To stop the session, run: pipeline.stop()


2026-10-12 14:13:42,437 | INFO | Calibration complete with 12 valid face samples


## 6. Inspect live status from the notebook

This cell shows recent observations in the notebook. Re-run it whenever you want to refresh the view. Streamlit is better for a continuously refreshing dashboard.

In [7]:
# Run this cell again to refresh a compact notebook status view.
snapshot = pipeline.snapshot()
print("Status:", snapshot["status"])
if snapshot["results"]:
    display(pd.DataFrame(snapshot["results"][-10:])[ ["captured_at", "index", "face_present", "person_count", "gaze_direction", "head_direction", "devices", "cues"] ])
if snapshot["batches"]:
    display(pd.DataFrame(snapshot["batches"][-5:]))


Status: {'running': True, 'calibration_complete': True, 'last_message': 'Photo analyzed', 'dropped_photos': 0}


,captured_at,index,face_present,person_count,gaze_direction,head_direction,devices,cues
0,2026-10-12T08:43:38.414561+00:00,7,True,1,UNKNOWN,UNKNOWN,[],[]
1,2026-10-12T08:43:39.167478+00:00,8,True,1,UNKNOWN,UNKNOWN,[],[]
2,2026-10-12T08:43:39.921613+00:00,9,True,1,UNKNOWN,UNKNOWN,[],[]
3,2026-10-12T08:43:40.674926+00:00,10,True,1,UNKNOWN,UNKNOWN,[],[]
4,2026-10-12T08:43:41.428878+00:00,11,True,1,UNKNOWN,UNKNOWN,[],[]
5,2026-10-12T08:43:42.431180+00:00,12,True,1,CENTER,CENTER,[],[]
6,2026-10-12T08:43:43.433498+00:00,13,True,1,CENTER,CENTER,[],[]
7,2026-10-12T08:43:44.436331+00:00,14,True,1,CENTER,CENTER,[],[]
8,2026-10-12T08:43:45.442816+00:00,15,True,1,CENTER,CENTER,[],[]
9,2026-10-12T08:43:46.445200+00:00,16,True,1,CENTER,LEFT,[],[head_turned]


,created_at,window_seconds,photos_analyzed,cue_counts,review_flag,status,note
0,2026-10-12T08:43:38.382050+00:00,5.0,2,{'second_person': 2},False,NO MULTI-CUE FLAG,Heuristic batch indicator; not a probability o...
1,2026-10-12T08:43:43.441567+00:00,5.0,10,{},False,NO MULTI-CUE FLAG,Heuristic batch indicator; not a probability o...


## 7. Stop and close

In [9]:
# Stop workers cleanly when the session is complete.
pipeline.stop()


2026-10-12 14:14:00,782 | INFO | Pipeline stopped


## Production-readiness notes

- **Tablet detection:** the default COCO model normally has `cell phone` and `laptop`, but no dependable `tablet` class. Use a validated custom model with a `tablet` label if tablet detection is a requirement.
- **Calibration:** this implementation calibrates landmark ratios; it does not estimate a medically/physically accurate 3D head pose. Poor lighting, glasses, camera angle, and face occlusion can reduce reliability.
- **Batch rule:** the example counts distinct cue types in each batch. It is intentionally transparent but not statistically calibrated. Tune it on labelled validation sessions; report false-positive and false-negative rates.
- **Fairness and review:** looking away can be normal. The overlay is only a reminder, and a batch flag must not automatically disqualify a candidate.
- **Evidence integrity:** JSONL and JPEG files are ordinary files, not tamper-proof evidence. For production, add access control, retention rules, checksums/signed manifests, consent, and audit logging.
- **Threading:** camera capture is paced by `sleep`, and model inference can take longer than one second. Queue drop counts and timestamps should be monitored to measure whether the system meets the required cadence.